# Sesión en vivo — inferencia causal

Ejecuta las celdas en orden durante la clase. Los 4 módulos comparten un
solo formulario de Web3Forms (2 campos: `modulo` y `payload_json`), así que
se leen de un único `data/responses.csv` (exportado del dashboard de
Web3Forms — ver `docs/DATA_BACKEND.md`) y se separan aquí por columna
`modulo`.

Si ese archivo todavía no existe, `SIMULATE = True` genera datos
**ficticios** solo para probar que el código corre antes de la clase real
— nunca los confundas con resultados reales, se marcan explícitamente.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path("data")
RESPONSES_CSV = DATA_DIR / "responses.csv"
SIMULATE = not RESPONSES_CSV.exists()

pd.set_option("display.width", 120)
rng = np.random.default_rng(7)

EDAD_MID = {"<20": 18, "20-24": 22, "25-29": 27, "30-39": 34.5, "40+": 45}

def load_module(modulo):
    """Lee data/responses.csv y junta las respuestas de este módulo, que
    pueden venir en dos formatos:
      - una fila por módulo (páginas individuales module-a.html..d.html),
        con Modulo == 'A'|'B'|'C'|'D' y una columna payload_json; o
      - una fila por persona (flow.html, los 4 módulos en un solo envío
        para no disparar el filtro antispam de Web3Forms al mandar 4
        POSTs separados), con Modulo == 'FLOW' y columnas
        payload_a_json..payload_d_json, una por módulo.
    Normaliza los nombres de columna porque el export de Web3Forms los
    entrega con mayúsculas/espacios ("Modulo", "Payload Json"), no
    snake_case."""
    df = pd.read_csv(RESPONSES_CSV)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    frames = []
    if "modulo" in df.columns and "payload_json" in df.columns:
        individual = df[df["modulo"].astype(str).str.upper() == modulo.upper()]
        if not individual.empty:
            frames.append(individual["payload_json"].apply(json.loads).apply(pd.Series))

    combined_col = f"payload_{modulo.lower()}_json"
    if combined_col in df.columns:
        combined = df[df[combined_col].notna()]
        if not combined.empty:
            frames.append(combined[combined_col].apply(json.loads).apply(pd.Series))

    if not frames:
        return pd.DataFrame()  # sin respuestas todavía para este módulo
    return pd.concat(frames, ignore_index=True)

if SIMULATE:
    print("Usando datos SIMULADOS de prueba -- no son respuestas reales.")


In [ ]:
def simulate_module_a(n=70):
    genero = rng.choice(["Mujer", "Hombre", "Otro"], n, p=[.48, .48, .04])
    edad = rng.choice(list(EDAD_MID), n)
    relig = rng.integers(1, 6, n)
    practica = rng.choice(["Sí", "No"], n)
    rol = rng.choice(["vendedor", "comprador"], n)
    es_vendedor = (rol == "vendedor").astype(int)
    es_hombre = (genero == "Hombre").astype(int)
    # Efecto dotación: vendedores (dueños, WTA) piden más que lo que
    # compradores (no dueños, WTP) pagarían por el mismo objeto. En esta
    # simulación la brecha es mayor entre hombres (heterogeneidad ilustrativa
    # -- la evidencia real sobre esto es mixta, es justamente lo que la
    # clase va a poder chequear con datos propios).
    base = 10
    brecha = 6 + 4 * es_hombre
    valor = np.clip(base + brecha * es_vendedor + rng.normal(0, 3, n), 1, None).round(1)
    return pd.DataFrame({
        "rol": rol, "valor_lucas": valor, "edad_rango": edad, "genero": genero,
        "religiosidad": relig, "practica_religion": practica,
    })

ITEMS_B = ["salud", "empleos", "bosque", "inversion", "estudiantes"]

def simulate_module_b(n=70):
    rows = []
    for _ in range(n):
        order = rng.permutation(len(ITEMS_B))
        edad = rng.choice(list(EDAD_MID)); genero = rng.choice(["Mujer", "Hombre", "Otro"], p=[.48, .48, .04])
        relig = rng.integers(1, 6); practica = rng.choice(["Sí", "No"])
        items = []
        for pos, idx in enumerate(order, start=1):
            item = ITEMS_B[idx]
            perdida = rng.random() < 0.5
            # el marco empuja hacia el riesgo en pérdida y hacia la certeza en ganancia;
            # ese jalón se atenúa (decae hacia 50/50) con la posición
            decay = max(0.15, 0.55 - 0.08 * (pos - 1))
            p_riesgo = 0.5 + (decay if perdida else -decay)
            eleccion = "B" if rng.random() < p_riesgo else "A"
            items.append({
                "item": item, "frame": "perdida" if perdida else "ganancia",
                "position": pos, "eleccion": eleccion,
            })
        rows.append({
            "items_json": json.dumps(items), "edad_rango": edad, "genero": genero,
            "religiosidad": relig, "practica_religion": practica,
        })
    return pd.DataFrame(rows)

TIPI_TRAITS = [
    ("extraversion", False), ("amabilidad", True), ("meticulosidad", False),
    ("estabilidad_emocional", True), ("apertura", False), ("extraversion", True),
    ("amabilidad", False), ("meticulosidad", True), ("estabilidad_emocional", False),
    ("apertura", True),
]

def simulate_module_c(n=70):
    rows = []
    for _ in range(n):
        variante = rng.choice(["switch", "push"])
        genero = rng.choice(["Mujer", "Hombre", "Otro"], p=[.48, .48, .04])
        relig = rng.integers(1, 6); practica = rng.choice(["Sí", "No"])
        base = 5.2 if variante == "switch" else 3.4  # push se juzga menos aceptable
        accept = np.clip(rng.normal(base, 1.3), 1, 7)
        tipi = [{"trait": t, "r": r, "value": int(rng.integers(1, 8))} for t, r in TIPI_TRAITS]
        rows.append({
            "variante": variante, "aceptabilidad": round(float(accept), 1),
            "genero": genero, "religiosidad": relig, "practica_religion": practica,
            "tipi_json": json.dumps(tipi),
        })
    return pd.DataFrame(rows)

def simulate_module_d(n=70):
    genero = rng.choice(["Mujer", "Hombre", "Otro"], n, p=[.48, .48, .04])
    edad = rng.choice(list(EDAD_MID), n)
    relig = rng.integers(1, 6, n)
    practica = rng.choice(["Sí", "No"], n)
    oferta = rng.choice(["justa", "injusta"], n)
    es_injusta = (oferta == "injusta").astype(int)
    es_mujer = (genero == "Mujer").astype(int)
    monto = np.where(oferta == "justa", 5, 2)
    # las ofertas injustas se rechazan mucho más que las justas; en esta
    # simulación el rechazo es algo mayor entre mujeres (heterogeneidad
    # ilustrativa -- de nuevo, algo que la clase puede chequear con datos propios)
    p_rechaza = 0.08 + es_injusta * (0.55 + 0.15 * es_mujer)
    decision = np.where(rng.random(n) < p_rechaza, "Rechaza", "Acepta")
    return pd.DataFrame({
        "oferta": oferta, "monto_propio_lucas": monto, "decision": decision,
        "edad_rango": edad, "genero": genero, "religiosidad": relig, "practica_religion": practica,
    })


## Módulo A — Vender o comprar

Efecto dotación (Kahneman, Knetsch & Thaler, 1990): cada estudiante es
**vendedor/a** (dueño/a del objeto, se le pregunta su WTA) o
**comprador/a** (no dueño/a, se le pregunta su WTP), asignado al azar.
Primero el ATE global (¿pesa ser dueño?), luego si esa brecha depende
del género.

In [ ]:
df_a = simulate_module_a() if SIMULATE else load_module("A")

if not SIMULATE and "rol" in df_a.columns:
    # Excluir filas del instrumento anterior (RUT/WTP), reconocibles porque
    # no tienen "rol" -- son de pruebas de antes del rediseño del Módulo A,
    # no respuestas al experimento actual.
    n_before = len(df_a)
    df_a = df_a[df_a["rol"].notna()].reset_index(drop=True)
    if n_before > len(df_a):
        print(f"({n_before - len(df_a)} fila(s) del instrumento anterior del Módulo A excluidas)")

df_a["valor_lucas"] = pd.to_numeric(df_a["valor_lucas"])
df_a["religiosidad"] = pd.to_numeric(df_a["religiosidad"])
df_a["edad_num"] = df_a["edad_rango"].map(EDAD_MID)
df_a["mujer"] = (df_a["genero"] == "Mujer").astype(int)
df_a["vendedor"] = (df_a["rol"] == "vendedor").astype(int)

# Para estimar el ATE hace falta ver ambos roles al menos un par de veces.
enough_a = df_a["rol"].nunique() >= 2 and len(df_a) >= 4
if not enough_a:
    print(f"Aún no hay suficiente variación en el Módulo A (n={len(df_a)}, "
          f"roles distintos: {df_a['rol'].nunique()}) para estimar el ATE. "
          "Las regresiones de abajo se saltan hasta que haya más respuestas.")
df_a.head()


In [ ]:
# Descriptivo primero: valor promedio declarado, por rol
fig, ax = plt.subplots(figsize=(5, 4))
df_a.groupby("rol")["valor_lucas"].mean().reindex(["comprador", "vendedor"]).plot(
    kind="bar", ax=ax, color=["#163C3A", "#2B6E6B"])
ax.set_ylabel("Valor promedio declarado (lucas)"); ax.set_xlabel("")
ax.set_title("¿Cuánto vale el mismo objeto según el rol asignado?")
plt.tight_layout(); plt.show()


In [ ]:
# ATE: ser vendedor/a (dueño/a) vs. comprador/a, sobre el valor declarado
if enough_a:
    ate_a = smf.ols("valor_lucas ~ vendedor", data=df_a).fit(cov_type="HC1")
    print(ate_a.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")


In [ ]:
def balance_table(df, split_col, label):
    """Compara covariables entre las dos mitades de `split_col` (bool o 0/1)."""
    print(f"--- Balance: {label} ---")
    if df[split_col].nunique() < 2:
        print(f"(sin datos suficientes todavía -- ambos grupos necesitan al menos 1 fila; n={len(df)})")
        return pd.DataFrame(columns=["Covariable", "Grupo 0", "Grupo 1", "Diferencia", "t"])
    mask = df[split_col].astype(bool)
    rows = []
    for col, name in [("edad_num", "Edad"), ("mujer", "% mujer"), ("religiosidad", "Religiosidad (1-5)")]:
        g0 = df.loc[~mask, col].astype(float)
        g1 = df.loc[mask, col].astype(float)
        diff = g1.mean() - g0.mean()
        se = (np.sqrt(g0.var(ddof=1) / len(g0) + g1.var(ddof=1) / len(g1))
              if len(g0) > 1 and len(g1) > 1 else np.nan)
        t = diff / se if se and se > 0 else np.nan
        rows.append({"Covariable": name, "Grupo 0": round(g0.mean(), 2), "Grupo 1": round(g1.mean(), 2),
                     "Diferencia": round(diff, 2), "t": round(t, 2) if pd.notna(t) else np.nan})
    return pd.DataFrame(rows)

balance_a = balance_table(df_a, "vendedor", "Módulo A -- asignación aleatoria (vendedor vs comprador)")
balance_a


### Heterogeneidad por género

¿La brecha vendedor/comprador (el efecto dotación) es del mismo tamaño
para hombres y mujeres, o no? Primero el gráfico, luego la regresión con
interacción.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
means_a = df_a.groupby(["genero", "rol"])["valor_lucas"].mean().unstack()
means_a = means_a.reindex(columns=["comprador", "vendedor"])
means_a.plot(kind="bar", ax=ax, color=["#163C3A", "#2B6E6B"])
ax.set_ylabel("Valor promedio declarado (lucas)"); ax.set_xlabel("Género")
ax.set_title("Efecto dotación por género")
ax.legend(title="Rol")
plt.tight_layout(); plt.show()


In [ ]:
# Interacción: ¿el ATE (ser vendedor/a) cambia con el género?
if enough_a and df_a["mujer"].nunique() >= 2:
    het_a = smf.ols("valor_lucas ~ vendedor * mujer", data=df_a).fit(cov_type="HC1")
    print(het_a.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")


## Módulo B — Ganar o perder

Efecto marco (Tversky & Kahneman, 1981, "Asian disease problem"): 5
dilemas de dominios distintos, cada uno descrito en términos de
**ganancia o pérdida asignados al azar** por el sitio, orden de
presentación también aleatorio. Primero el ATE global y por posición
(para ver si el efecto decae con la repetición); luego el mismo balance
de covariables que en el Módulo A, para comparar.

In [ ]:
df_b_wide = simulate_module_b() if SIMULATE else load_module("B")

if len(df_b_wide) == 0:
    df_b = pd.DataFrame()
    enough_b = False
    print("Aún no hay respuestas para el Módulo B todavía.")
else:
    if "submission_id" not in df_b_wide.columns:  # por si faltara en algún envío viejo
        df_b_wide = df_b_wide.reset_index(drop=True).reset_index().rename(columns={"index": "submission_id"})
    df_b_wide["religiosidad"] = pd.to_numeric(df_b_wide["religiosidad"])
    df_b_wide["edad_num"] = df_b_wide["edad_rango"].map(EDAD_MID)
    df_b_wide["mujer"] = (df_b_wide["genero"] == "Mujer").astype(int)

    # Expandir items_json a formato largo: 5 filas por respuesta
    long_rows = []
    for _, row in df_b_wide.iterrows():
        for item in json.loads(row["items_json"]):
            long_rows.append({**item, "submission_id": row["submission_id"]})
    df_b = pd.DataFrame(long_rows)
    enough_b = len(df_b) >= 10  # ~2 respuestas completas como mínimo para que algo de esto tenga sentido
    if not enough_b:
        print(f"Aún no hay suficientes respuestas para el Módulo B (filas: {len(df_b)}). "
              "Las celdas de abajo se saltan hasta que haya más.")
df_b.head()


In [ ]:
# Descriptivo primero: proporción que elige la opción riesgosa (B), por marco
if enough_b:
    df_b["perdida"] = (df_b["frame"] == "perdida").astype(int)
    df_b["riesgo"] = (df_b["eleccion"] == "B").astype(int)

    p_global = df_b.groupby("frame")["riesgo"].mean()
    print("Proporción que elige la opción riesgosa, por marco:")
    print(p_global)

    p_by_pos = df_b.groupby(["position", "frame"])["riesgo"].mean().unstack()
    p_by_pos["brecha"] = p_by_pos["perdida"] - p_by_pos["ganancia"]
    print(p_by_pos)

    fig, ax = plt.subplots(figsize=(6, 4))
    p_by_pos["brecha"].plot(marker="o", ax=ax, color="#B9862A")
    ax.axhline(0, color="grey", lw=.8)
    ax.set_xlabel("Posición en la secuencia (1-5)"); ax.set_ylabel("Brecha (% riesgo, pérdida − ganancia)")
    ax.set_title("¿El efecto marco decae con la repetición?")
    plt.tight_layout(); plt.show()
else:
    print("(sin datos suficientes todavía)")


In [ ]:
# Regresión con efectos fijos por ítem y errores estándar agrupados por estudiante
if enough_b:
    model_b = smf.ols(
        "riesgo ~ perdida + position + perdida:position + C(item)", data=df_b
    ).fit(cov_type="cluster", cov_kwds={"groups": df_b["submission_id"]})
    print(model_b.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")


In [ ]:
if enough_b:
    df_b_wide["perdida_prom"] = df_b.groupby("submission_id")["perdida"].mean().reindex(df_b_wide["submission_id"]).values
    df_b_wide["mitad_perdida"] = df_b_wide["perdida_prom"] >= df_b_wide["perdida_prom"].median()
    balance_b = balance_table(df_b_wide, "mitad_perdida", "Módulo B -- asignación aleatoria del marco (SÍ es aleatorización del investigador)")
else:
    balance_b = None
    print("(sin datos suficientes todavía)")
balance_b


### A vs. B, lado a lado

Dos asignaciones aleatorias independientes (vendedor/comprador en A,
marco ganancia/pérdida en B) -- ambas deberían salir balanceadas en
expectativa, en variables que medimos *y* en las que no. Verlo dos veces,
en dos preguntas de investigación distintas, es la evidencia repetida de
que la aleatorización -- y no el diseño del instrumento -- es lo que hace
el trabajo.

In [ ]:
if balance_b is not None:
    display(pd.concat([balance_a.set_index("Covariable"), balance_b.set_index("Covariable")],
                       axis=1, keys=["Módulo A (vendedor/comprador)", "Módulo B (marco ganancia/pérdida)"]))
else:
    print("Todavía no hay balance del Módulo B para comparar -- solo Módulo A por ahora:")
    display(balance_a)


## Módulo C — El dilema y tú

Variante del tranvía asignada al azar (switch = palanca, push = empujar).
ATE global sobre la aceptabilidad (1-7), y heterogeneidad por género,
religiosidad y personalidad (TIPI-10).

In [ ]:
df_c = simulate_module_c() if SIMULATE else load_module("C")
df_c["aceptabilidad"] = pd.to_numeric(df_c["aceptabilidad"])
df_c["religiosidad"] = pd.to_numeric(df_c["religiosidad"])
df_c["push"] = (df_c["variante"] == "push").astype(int)
df_c["mujer"] = (df_c["genero"] == "Mujer").astype(int)
df_c["religioso"] = (df_c["religiosidad"].astype(float) >= 4).astype(int)

def score_tipi(tipi_json):
    items = json.loads(tipi_json)
    scores = {}
    for it in items:
        v = 8 - it["value"] if it["r"] else it["value"]
        scores.setdefault(it["trait"], []).append(v)
    return {t: np.mean(v) for t, v in scores.items()}

tipi_scores = df_c["tipi_json"].apply(score_tipi).apply(pd.Series)
df_c = pd.concat([df_c, tipi_scores], axis=1)

# Para estimar push ~ variante hace falta ver ambas variantes al menos un par de veces cada una.
enough_c = df_c["push"].nunique() >= 2 and len(df_c) >= 4
if not enough_c:
    print(f"Aún no hay suficiente variación en el Módulo C (n={len(df_c)}, "
          f"variantes distintas: {df_c['push'].nunique()}) para estimar el ATE. "
          "Las regresiones de abajo se saltan hasta que haya más respuestas.")
df_c.head()


In [ ]:
# Descriptivo primero: aceptabilidad promedio por variante, separado por género
# (antes de cualquier regresión -- ¿se ve una brecha a simple vista?)
fig, ax = plt.subplots(figsize=(6, 4))
means = df_c.groupby(["genero", "variante"])["aceptabilidad"].mean().unstack()
means = means.reindex(columns=["switch", "push"])
means.plot(kind="bar", ax=ax, color=["#B78BC0", "#4A2E52"])
ax.set_ylabel("Aceptabilidad promedio (1-7)"); ax.set_xlabel("Género")
ax.set_title("Aceptabilidad por variante del dilema, por género")
ax.legend(title="Variante")
plt.tight_layout(); plt.show()


In [ ]:
# ATE global: empujar vs. jalar la palanca
if enough_c:
    ate_c = smf.ols("aceptabilidad ~ push", data=df_c).fit(cov_type="HC1")
    print(ate_c.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")


In [ ]:
df_c["consciente_alto"] = (df_c["meticulosidad"] >= df_c["meticulosidad"].median()).astype(int)

if enough_c:
    subgroups = {
        "Mujer": "mujer",
        "Religioso/a (4-5)": "religioso",
        "Meticulosidad alta": "consciente_alto",
    }
    rows = []
    for label, col in subgroups.items():
        m = smf.ols(f"aceptabilidad ~ push * {col}", data=df_c).fit(cov_type="HC1")
        rows.append({"Subgrupo": label, "ATE (push)": m.params["push"],
                     "Interacción": m.params[f"push:{col}"], "p (interacción)": m.pvalues[f"push:{col}"]})
    het_table = pd.DataFrame(rows)
else:
    het_table = None
    print("(sin datos suficientes todavía)")
het_table


In [ ]:
if het_table is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(het_table["Interacción"], het_table["Subgrupo"], fmt="o", color="#6E4A76")
    ax.axvline(0, color="grey", lw=.8)
    ax.set_xlabel("Interacción push × subgrupo (CATE - ATE)")
    ax.set_title("Heterogeneidad del efecto por subgrupo")
    plt.tight_layout(); plt.show()
else:
    print("(sin datos suficientes todavía)")


## Módulo D — Aceptar o rechazar

Juego del ultimátum (Güth, Schmittberger & Schwarze, 1982): oferta
**asignada al azar** -- justa (5/5) o injusta (2/8) sobre 10 lucas.
¿Se rechazan las ofertas injustas aunque eso signifique quedarse sin
nada? Y ¿ese rechazo depende del género?

In [ ]:
df_d = simulate_module_d() if SIMULATE else load_module("D")
df_d["religiosidad"] = pd.to_numeric(df_d["religiosidad"])
df_d["monto_propio_lucas"] = pd.to_numeric(df_d["monto_propio_lucas"])
df_d["edad_num"] = df_d["edad_rango"].map(EDAD_MID)
df_d["mujer"] = (df_d["genero"] == "Mujer").astype(int)
df_d["injusta"] = (df_d["oferta"] == "injusta").astype(int)
df_d["rechaza"] = (df_d["decision"] == "Rechaza").astype(int)

# Para estimar el ATE hace falta ver ambos tipos de oferta al menos un par de veces.
enough_d = df_d["oferta"].nunique() >= 2 and len(df_d) >= 4
if not enough_d:
    print(f"Aún no hay suficiente variación en el Módulo D (n={len(df_d)}, "
          f"tipos de oferta distintos: {df_d['oferta'].nunique()}) para estimar el ATE. "
          "Las regresiones de abajo se saltan hasta que haya más respuestas.")
df_d.head()


In [ ]:
# Descriptivo primero: proporción que rechaza, por tipo de oferta
fig, ax = plt.subplots(figsize=(5, 4))
df_d.groupby("oferta")["rechaza"].mean().reindex(["justa", "injusta"]).plot(
    kind="bar", ax=ax, color=["#6FA3D8", "#1B3A57"])
ax.set_ylabel("Proporción que rechaza"); ax.set_xlabel(""); ax.set_ylim(0, 1)
ax.set_title("¿Se rechazan más las ofertas injustas?")
plt.tight_layout(); plt.show()


In [ ]:
# ATE: oferta injusta vs. justa, sobre la probabilidad de rechazar
if enough_d:
    ate_d = smf.ols("rechaza ~ injusta", data=df_d).fit(cov_type="HC1")
    print(ate_d.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")


In [ ]:
balance_d = balance_table(df_d, "injusta", "Módulo D -- asignación aleatoria (oferta justa vs injusta)")
balance_d


### Heterogeneidad por género

¿El rechazo a ofertas injustas es del mismo tamaño para hombres y
mujeres, o no?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
means_d = df_d.groupby(["genero", "oferta"])["rechaza"].mean().unstack()
means_d = means_d.reindex(columns=["justa", "injusta"])
means_d.plot(kind="bar", ax=ax, color=["#6FA3D8", "#1B3A57"])
ax.set_ylabel("Proporción que rechaza"); ax.set_xlabel("Género"); ax.set_ylim(0, 1)
ax.set_title("Rechazo de ofertas injustas, por género")
ax.legend(title="Oferta")
plt.tight_layout(); plt.show()


In [ ]:
# Interacción: ¿el ATE (oferta injusta) cambia con el género?
if enough_d and df_d["mujer"].nunique() >= 2:
    het_d = smf.ols("rechaza ~ injusta * mujer", data=df_d).fit(cov_type="HC1")
    print(het_d.summary().tables[1])
else:
    print("(sin datos suficientes todavía)")
